# Intelligent RAG-Based Chatbot
### Building a Retrieval-Augmented Generation System with LangChain and Google Gemini

---

**Project:** Building an Intelligent RAG-based Chatbot using LangChain and LLMs  
**Stack:** LangChain · Google Gemini (LLM + Embeddings) · FAISS · ipywidgets

---

## Problem Domain

Traditional search systems return links — users still have to read through documents manually. Large Language Models (LLMs) can answer questions fluently but hallucinate facts they were not trained on. **Retrieval-Augmented Generation (RAG)** solves both problems: it retrieves the most relevant passages from a private knowledge base and feeds them as context to the LLM, producing accurate, grounded answers with source attribution.

This chatbot is designed as a **document Q&A assistant** — upload any combination of PDFs, Word documents, plain-text, or Markdown files, and the system answers questions strictly from that content.

---

## System Architecture

```
┌──────────────────────────────────────────────────────────────────┐
│                        RAG PIPELINE                              │
│                                                                  │
│  ┌─────────────┐   ┌──────────────┐   ┌────────────────────┐   │
│  │  INGESTION  │   │  EMBEDDING   │   │    RETRIEVAL       │   │
│  │             │   │              │   │                    │   │
│  │ PDF/DOCX/   │──▶│ Gemini       │──▶│ FAISS Index        │   │
│  │ TXT/MD      │   │ text-embed   │   │ Cosine Similarity  │   │
│  │ ↓ Chunks    │   │ -004         │   │ Top-K Documents    │   │
│  │ (1000/200)  │   │              │   │                    │   │
│  └─────────────┘   └──────────────┘   └────────────────────┘   │
│                                                   │             │
│  ┌─────────────────────────────────────────────── ▼ ──────────┐ │
│  │                    GENERATION                              │ │
│  │  Conversation History + Retrieved Context + User Question  │ │
│  │                         ↓                                  │ │
│  │              Google Gemini 1.5 Flash (LLM)                 │ │
│  │                         ↓                                  │ │
│  │           Grounded Answer + Source Attribution             │ │
│  └────────────────────────────────────────────────────────────┘ │
└──────────────────────────────────────────────────────────────────┘
```

**Components:**
- `DocumentProcessor` — loads, chunks, embeds, and indexes documents into FAISS
- `SessionMemory` — maintains a sliding window of the last N conversation turns
- `RAGEngine` — orchestrates retrieval, prompt assembly, and LLM invocation
- `ChatUI` — interactive ipywidgets interface (upload, index, chat, settings)


---
## Cell 1 — Install Dependencies

Run once. Uses `uv` for fast resolution, with a `pip` fallback.

In [1]:
import subprocess, sys

PACKAGES = [
    # LangChain core + integrations
    "langchain>=0.3",
    "langchain-core>=0.3",
    "langchain-community>=0.3",
    "langchain-google-genai>=2.0",   # Gemini LLM + embeddings
    "langchain-huggingface",          # fallback embedding option
    "langchain-text-splitters",
    # Vector store
    "faiss-cpu",
    # Document loaders
    "pypdf",
    "docx2txt",
    # Utilities
    "python-dotenv",
    "google-generativeai",
    # UI
    "ipywidgets>=8.0",
]

def install(packages):
    # Try uv first (fast), then fall back to pip
    for pkg in packages:
        result = subprocess.run(
            ["uv", "pip", "install", pkg],
            capture_output=True, text=True
        )
        if result.returncode != 0:
            # uv not available or failed — use pip
            result = subprocess.run(
                [sys.executable, "-m", "pip", "install", pkg,
                 "--break-system-packages", "-q"],
                capture_output=True, text=True
            )
        status = "✅" if result.returncode == 0 else f"❌  {result.stderr.strip()[:80]}"
        print(f"  {pkg:<40} {status}")

print("Installing packages (uv → pip fallback)...\n")
install(PACKAGES)
print("\nAll dependencies installed.")

Installing packages (uv → pip fallback)...

  langchain>=0.3                           ✅
  langchain-core>=0.3                      ✅
  langchain-community>=0.3                 ✅
  langchain-google-genai>=2.0              ✅
  langchain-huggingface                    ✅
  langchain-text-splitters                 ✅
  faiss-cpu                                ✅
  pypdf                                    ✅
  docx2txt                                 ✅
  python-dotenv                            ✅
  google-generativeai                      ✅
  ipywidgets>=8.0                          ✅

All dependencies installed.


---
## Cell 2 — API Key Configuration

In [2]:
import os
from dotenv import load_dotenv

# Priority 1: .env file in the project root
load_dotenv()

# Priority 2: prompt the user (value is masked and not stored in history)
if not os.getenv("GEMINI_API_KEY"):
    import getpass
    os.environ["GEMINI_API_KEY"] = getpass.getpass(
        "Enter your Gemini API Key (get one free at https://aistudio.google.com): "
    )

print("API key configured.")
print("Tip: create a .env file with GEMINI_API_KEY=... to skip this step next time.")

API key configured.
Tip: create a .env file with GEMINI_API_KEY=... to skip this step next time.


---
## Cell 3 — RAG Pipeline

In [ ]:
# =============================================================================
#  RAG PIPELINE
#  Three classes with single responsibilities:
#    DocumentProcessor  — ingest, chunk, embed, index
#    SessionMemory      — sliding-window conversation history
#    RAGEngine          — retrieval + prompt assembly + LLM call
# =============================================================================

import os
import shutil
import tempfile
from pathlib import Path
from typing import Optional

from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_community.document_loaders import (
    PyPDFLoader,
    TextLoader,
    Docx2txtLoader,
)
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_google_genai import (
    ChatGoogleGenerativeAI,
    GoogleGenerativeAIEmbeddings,   # Gemini embedding model
)
from langchain_huggingface import HuggingFaceEmbeddings


# -----------------------------------------------------------------------------
#  1. DocumentProcessor
# -----------------------------------------------------------------------------
class DocumentProcessor:
    """
    Handles the ingestion half of the RAG pipeline.

    Steps:
        load_file() / load_directory()  →  RecursiveCharacterTextSplitter
        →  GoogleGenerativeAIEmbeddings  →  FAISS index

    Each document chunk is tagged with its source filename in metadata so the
    retriever can surface provenance alongside the answer.
    """

    LOADER_MAP = {
        ".pdf":  PyPDFLoader,
        ".txt":  TextLoader,
        ".docx": Docx2txtLoader,
        ".md":   TextLoader,
    }

    def __init__(
        self,
        chunk_size: int = 1000,
        chunk_overlap: int = 200,
        retriever_k: int = 4,
    ):
        self.chunk_size    = chunk_size
        self.chunk_overlap = chunk_overlap
        self.retriever_k   = retriever_k

        self.documents:    list[Document] = []
        self.chunks:       list[Document] = []
        self.embeddings:   Optional[GoogleGenerativeAIEmbeddings] = None
        self.vector_store: Optional[FAISS] = None
        self.loaded_files: list[str] = []

    # -- Loading -------------------------------------------------------------- #

    def load_file(self, file_path: str) -> list[Document]:
        """Load a single supported file and append its pages/sections."""
        path   = Path(file_path)
        suffix = path.suffix.lower()

        if suffix not in self.LOADER_MAP:
            raise ValueError(
                f"Unsupported file type '{suffix}'. "
                f"Supported: {list(self.LOADER_MAP.keys())}"
            )

        loader = self.LOADER_MAP[suffix](str(path))
        docs   = loader.load()

        # Tag each page/section with the source filename
        for doc in docs:
            doc.metadata["source_file"] = path.name

        self.documents.extend(docs)
        if path.name not in self.loaded_files:
            self.loaded_files.append(path.name)

        return docs

    def load_directory(self, dir_path: str) -> list[Document]:
        """Recursively load all supported files from a folder."""
        all_docs = []
        for file in Path(dir_path).rglob("*"):
            if file.is_file() and file.suffix.lower() in self.LOADER_MAP:
                try:
                    all_docs.extend(self.load_file(str(file)))
                    print(f"  Loaded: {file.name}")
                except Exception as exc:
                    print(f"  Error loading {file.name}: {exc}")
        return all_docs

    # -- Chunking ------------------------------------------------------------- #

    def _split(self) -> list[Document]:
        splitter = RecursiveCharacterTextSplitter(
            chunk_size    = self.chunk_size,
            chunk_overlap = self.chunk_overlap,
            separators    = ["\n\n", "\n", " ", ""],
        )
        self.chunks = splitter.split_documents(self.documents)
        return self.chunks

    # -- Embeddings ---------------------------------------------------------- #

    # def _get_embeddings(self) -> GoogleGenerativeAIEmbeddings:
    #     """
    #     Returns a cached GoogleGenerativeAIEmbeddings instance.
    #     Model: gemini-embedding-001  (1536-dim, best accuracy in Gemini family).
    #     """
    #     if self.embeddings is None:
    #         api_key = os.getenv("GEMINI_API_KEY")
    #         if not api_key:
    #             raise EnvironmentError("GEMINI_API_KEY not set. Run Cell 2 first.")
    #         self.embeddings = GoogleGenerativeAIEmbeddings(
    #             model = "gemini-embedding-001",
    #             task_type="retrieval_document" ,
    #             google_api_key = api_key,
    #         )
    #     return self.embeddings

    def _get_embeddings(self):
        """
        Returns a cached HuggingFace embedding model.
        """
        if self.embeddings is None:
            self.embeddings = HuggingFaceEmbeddings(
                model_name="sentence-transformers/all-MiniLM-L6-v2"
            )
            
        return self.embeddings

    # -- Vector Store --------------------------------------------------------- #


    def build_vector_store(self) -> FAISS:
        """
        Split loaded documents into chunks, embed them with Gemini, and
        build a FAISS index for fast similarity search.
        """
        if not self.documents:
            raise ValueError(
                "No documents loaded. Call load_file() or load_directory() first."
            )
        chunks     = self._split()
        embeddings = self._get_embeddings()
        print(f"  Indexing {len(chunks)} chunks...")
        self.vector_store = FAISS.from_documents(chunks, embeddings)
        print("  Vector store ready.")
        return self.vector_store

    def add_documents(self, new_docs: list[Document]) -> None:
        """Incrementally add documents to an existing FAISS index."""

        splitter = RecursiveCharacterTextSplitter(
            chunk_size    = self.chunk_size,
            chunk_overlap = self.chunk_overlap,
            separators=["\n\n", "\n", " ", ""]
        )
        new_chunks = splitter.split_documents(new_docs)
        self.chunks.extend(new_chunks)
        embeddings = self._get_embeddings()

        if self.vector_store is None:
            self.vector_store = FAISS.from_documents(new_chunks, embeddings)
        else:
            self.vector_store.add_documents(new_chunks)

    def get_retriever(self):
        """Return a LangChain retriever ready for use in the RAG chain."""
        if self.vector_store is None:
            raise ValueError("Vector store not built yet. Call build_vector_store().")
        return self.vector_store.as_retriever(
            search_type   = "similarity",
            search_kwargs = {"k": self.retriever_k},
        )


# -----------------------------------------------------------------------------
#  2. SessionMemory
# -----------------------------------------------------------------------------
class SessionMemory:
    """
    Sliding-window conversation history.

    Stores the full message list in an InMemoryChatMessageHistory (LangChain
    Core — no deprecated langchain.memory dependency). Only the last `window_k`
    human/AI exchange pairs are injected into the prompt to keep token counts
    predictable while still supporting follow-up questions.

    history  — raw list of dicts used by the UI renderer
    """

    def __init__(self, window_k: int = 5):
        self.window_k = window_k
        self._store   = InMemoryChatMessageHistory()
        self.history: list[dict] = []

    def add(self, question: str, answer: str) -> None:
        """Append a human/AI exchange to the history."""
        self._store.add_user_message(question)
        self._store.add_ai_message(answer)
        self.history.append({"role": "user",      "content": question})
        self.history.append({"role": "assistant", "content": answer})



    def clear(self) -> None:
        """Reset history for a new chat session."""
        self._store.clear()
        self.history.clear()


# -----------------------------------------------------------------------------
#  3. RAGEngine
# -----------------------------------------------------------------------------
class RAGEngine:
    """
    Orchestrates the full retrieval-augmented generation cycle.

    On each call to ask():
        1. Retrieve top-K relevant chunks from FAISS.
        2. Assemble a prompt: conversation window + retrieved context + question.
        3. Call Gemini 1.5 Flash and return the answer with source provenance.

    The LLM is instantiated once at construction time (not on every call) to
    avoid repeated authentication overhead.
    """

    PROMPT = PromptTemplate(
        input_variables=["chat_history", "context", "question"],
        template="""
You are a knowledgeable assistant with access to a private document library.

RULES:
1. Answer using ONLY the information in the CONTEXT section.
2. If the context does not contain enough information to answer, say:
   "The uploaded documents do not cover this topic."
   Then answer from your general knowledge, clearly prefixed with:
   "Based on general knowledge:"
3. Keep answers clear, well-structured, and factual.
4. When helpful, mention which document the information comes from.

CONVERSATION HISTORY:
{chat_history}

CONTEXT FROM DOCUMENTS:
{context}

QUESTION: {question}

ANSWER:


""",
    )

    def __init__(
        self,
        retriever,
        memory:      SessionMemory,
        model:       str   = "gemini-3.5-flash",
        temperature: float = 0.3,
    ):
        self.retriever = retriever
        self.memory    = memory

        api_key = os.getenv("GEMINI_API_KEY")
        if not api_key:
            raise EnvironmentError("GEMINI_API_KEY not set. Run Cell 2 first.")

        self.llm = ChatGoogleGenerativeAI(
            model          = model,
            google_api_key = api_key,
            temperature    = temperature,
        )

    def ask(self, question: str) -> dict:
        """
        Run the full RAG cycle and return a dict with keys:
            answer  — the LLM's response string
            sources — list of unique source filenames cited
            context — raw retrieved context (for debugging)
        """
        # Step 1: Retrieve
        retrieved = self.retriever.invoke(question)

        context_parts = []
        source_set = set()
        for doc in retrieved:
            source_file = doc.metadata.get('source_file', 'unknown')
            page_number = doc.metadata.get('page')

            if page_number is not None:
                context_parts.append(f"[Source: {source_file}, Page: {page_number}]\n{doc.page_content}")
                source_set.add(f"{source_file} (Page {page_number})")
            else:
                context_parts.append(f"[Source: {source_file}]\n{doc.page_content}")
                source_set.add(source_file)

        context = "\n\n".join(context_parts)
        sources = list(source_set)

        # Step 2: Assemble prompt
        filled = self.PROMPT.invoke({
            "chat_history": self.memory.get_window_string(),
            "context":      context,
            "question":     question,
        })

        # Step 3: Generate
        response = self.llm.invoke(filled)
        answer   = response.content[0]['text']

        if "The uploaded documents do not cover this topic." in answer:
          sources = None

        # Step 4: Persist to memory
        self.memory.add(question, answer)

        return {"answer": answer, "sources": sources, "context": context}


print("RAG pipeline classes loaded.")


C:\Users\karel\AppData\Local\Temp\ipykernel_17212\2651915453.py:19: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import (


RAG pipeline classes loaded.


---
## Cell 4 — Chat Interface (ipywidgets)

In [4]:
# =============================================================================
#  CHAT INTERFACE
#  Pure ipywidgets — works inside any Jupyter environment without Streamlit.
#
#  Features:
#    - FileUpload widget → writes to a tempfile.mkdtemp() session folder
#    - Incremental indexing: adding more files merges into the existing index
#    - Dark-themed chat bubbles with source-file attribution pills
#    - Settings accordion: temperature, retrieval K, memory window
#    - "New Chat" clears memory without losing the index
#    - "Clear Files" wipes the temp folder and resets everything
#    - Cell 5 (cleanup) permanently deletes the temp folder on exit
# =============================================================================

import os, shutil
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, HTML
import re

# ── Session state ──────────────────────────────────────────────────────────────
import tempfile
SESSION_DIR = tempfile.mkdtemp(prefix="rag_session_")
processor   = DocumentProcessor()
memory      = SessionMemory(window_k=5)
engine      = None          # created after first successful indexing
chat_html_parts: list[str] = []

print(f"Session folder: {SESSION_DIR}")


# ── Styles ────────────────────────────────────────────────────────────────────
CSS = """
<link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/katex@0.16.0/dist/katex.min.css">
<script defer src="https://cdn.jsdelivr.net/npm/katex@0.16.0/dist/katex.min.js"></script>
<script defer src="https://cdn.jsdelivr.net/npm/katex@0.16.0/dist/contrib/auto-render.min.js"
        onload="renderMathInElement(document.body, {delimiters: [{left: '$$', right: '$$', display: true}, {left: '$', right: '$', display: false}]});"></script>
<style>
.ragbox {
    background:#0f0f1a; border:1px solid #2a2a4a; border-radius:12px;
    padding:16px; height:440px; overflow-y:auto;
    font-family:'Segoe UI',system-ui,sans-serif; font-size:14px;
    line-height:1.65; scroll-behavior:smooth;
}
.msg-u {
    background:linear-gradient(135deg,#4f46e5,#7c3aed); color:#fff;
    border-radius:18px 18px 4px 18px; padding:10px 16px;
    margin:8px 0 8px 22%; box-shadow:0 2px 8px rgba(79,70,229,.4);
}
.msg-b {
    background:#1a1a2e; color:#e2e8f0;
    border:1px solid #2a2a5a; border-radius:18px 18px 18px 4px;
    padding:10px 16px; margin:8px 22% 8px 0;
    box-shadow:0 2px 6px rgba(0,0,0,.35);
}
.lbl { font-size:11px; font-weight:700; letter-spacing:.05em;
       margin-bottom:4px; opacity:.65; }
.src { font-size:11px; color:#818cf8; margin-top:6px; font-style:italic; }
.pill {
    display:inline-block; background:#1e1e3a; border:1px solid #4f46e5;
    color:#a5b4fc; border-radius:999px; padding:2px 10px;
    margin:2px; font-size:12px;
}
.err { color:#f87171; }
.msg-b h3, .msg-b h4 { color:#a5b4fc; margin:8px 0 4px 0; font-size:14px; font-weight:600; }
.msg-b ul { margin:6px 0; padding-left:24px; }
.msg-b li { margin:3px 0; color:#e2e8f0; }
.msg-b strong { color:#4ade80; font-weight:700; }
.msg-b em { color:#c7d2fe; font-style:italic; }
.msg-b code { background:#0f0f1a; padding:3px 6px; border-radius:4px; color:#f87171; font-size:12px; font-family:monospace; }
.msg-b .katex { font-size:1.1em; }
.msg-b table { margin:8px 0; border-collapse:collapse; font-size:13px; }
.msg-b th, .msg-b td { border:1px solid #2a2a5a; padding:6px 10px; text-align:left; }
.msg-b th { background:#1e1e3a; color:#a5b4fc; font-weight:600; }
</style>
"""


# ── Helpers ───────────────────────────────────────────────────────────────────
def _render(chat_widget):
    """Re-draw the chat box from the parts list."""
    if not chat_html_parts:
        inner = (
            "<div style='color:#475569;text-align:center;padding:60px 0 40px;'>"
            "Upload documents, click <b>Index</b>, then ask a question.</div>"
        )
    else:
        inner = "".join(chat_html_parts)
    chat_widget.value = f"<div class='ragbox'>{inner}</div>"


def _esc(text: str) -> str:
    return text.replace("&","&amp;").replace("<","&lt;").replace(">","&gt;")


def _format_markdown_to_html(text: str) -> str:
    """Convert markdown formatting to HTML while preserving LaTeX."""
    # Protect LaTeX blocks from markdown processing
    latex_blocks = []

    def protect_latex(match):
        latex_blocks.append(match.group(0))
        return f"__LATEX_{len(latex_blocks)-1}__"

    # Protect $...$ and $$...$$
    text = re.sub(r'\$\$[^$]+\$\$', protect_latex, text)  # $$ ... $$
    text = re.sub(r'\$[^$]+\$', protect_latex, text)       # $ ... $

    # Escape HTML first
    text = _esc(text)

    # Convert **bold** to <strong>
    text = re.sub(r'\*\*(.+?)\*\*', r'<strong>\1</strong>', text)

    # Convert *italic* to <em>
    text = re.sub(r'\*(.+?)\*', r'<em>\1</em>', text)

    # Convert `code` to <code>
    text = re.sub(r'`([^`]+)`', r'<code>\1</code>', text)

    # Convert ### headers to <h3>
    text = re.sub(r'^### (.+?)$', r'<h3>\1</h3>', text, flags=re.MULTILINE)
    text = re.sub(r'^## (.+?)$', r'<h4>\1</h4>', text, flags=re.MULTILINE)

    # Convert newlines to <br>
    text = text.replace('\n', '<br>')

    # Convert bullet lists (* item)
    text = re.sub(r'<br>\* ', '<br><li style="margin-left:-16px;"> ', text)
    text = re.sub(r'^<br><li', '<ul><li', text)

    # Close list tags
    if '<li' in text and '</ul>' not in text:
        text += '</ul>'

    # Restore LaTeX blocks but strip their delimiters (as KaTeX is not auto-rendering dynamically)
    for i, original_latex_string in enumerate(latex_blocks):
        content_without_delimiters = original_latex_string
        if original_latex_string.startswith('$$') and original_latex_string.endswith('$$'):
            content_without_delimiters = original_latex_string[2:-2]
        elif original_latex_string.startswith('$') and original_latex_string.endswith('$'):
            content_without_delimiters = original_latex_string[1:-1]

        text = text.replace(f'__LATEX_{i}__', content_without_delimiters)

    return text


def _push_user(text, chat_widget):
    chat_html_parts.append(
        f"<div class='msg-u'><div class='lbl'>You</div>{_esc(text)}</div>"
    )
    _render(chat_widget)


def _push_bot(text, sources, chat_widget):
    src_html = ""
    if sources:
        pills = "".join(f"<span class='pill'>📄 {_esc(s)}</span>" for s in sources)
        src_html = f"<div class='src'>Sources: {pills}</div>"

    # Format markdown and preserve LaTeX
    body = _format_markdown_to_html(text)

    chat_html_parts.append(
        f"<div class='msg-b'><div class='lbl'>Assistant</div>{body}{src_html}</div>"
    )
    _render(chat_widget)


def _push_err(text, chat_widget):
    chat_html_parts.append(
        f"<div class='msg-b'><div class='lbl err'>Error</div>"
        f"<span class='err'>{_esc(text)}</span></div>"
    )
    _render(chat_widget)


# ── Widget factory ────────────────────────────────────────────────────────────
def _make_widgets():
    L = widgets.Layout

    # Header
    header = widgets.HTML(
        "<div style='background:linear-gradient(135deg,#4f46e5,#7c3aed);"
        "border-radius:10px;padding:18px 22px;margin-bottom:10px;'>"
        "<div style='color:#fff;font-size:20px;font-weight:700;font-family:Segoe UI,sans-serif;'>"
        "RAG Chatbot</div>"
        "<div style='color:#c7d2fe;font-size:12px;margin-top:3px;'>"
        "Powered by Google Gemini · LangChain · FAISS</div></div>"
    )

    # Settings
    temp_sl = widgets.IntSlider(
        value=3, min=0, max=10, step=1, description="Temperature (×0.1):",
        style={"description_width":"160px"}, layout=L(width="420px")
    )
    k_sl = widgets.IntSlider(
        value=4, min=1, max=10, step=1, description="Top-K Retrieval:",
        style={"description_width":"160px"}, layout=L(width="420px")
    )
    mem_sl = widgets.IntSlider(
        value=5, min=1, max=20, step=1, description="Memory Window (turns):",
        style={"description_width":"160px"}, layout=L(width="420px")
    )
    chunk_sl = widgets.IntSlider(
        value=1000, min=200, max=3000, step=100, description="Chunk Size:",
        style={"description_width":"160px"}, layout=L(width="420px")
    )
    accordion = widgets.Accordion(
        children=[widgets.VBox([temp_sl, k_sl, mem_sl, chunk_sl],
            layout=L(padding="12px"))]
    )
    accordion.set_title(0, "Advanced Settings")
    accordion.selected_index = None

    # File upload section
    upload_btn = widgets.FileUpload(
        accept=".pdf,.txt,.docx,.md", multiple=True,
        description="Upload Files",
        layout=L(width="180px"),
        style={"button_color":"#4f46e5"},
    )
    index_btn = widgets.Button(
        description="Index Documents",
        layout=L(width="160px", height="36px"),
        style={"button_color":"#059669", "font_weight":"bold"},
    )
    clear_files_btn = widgets.Button(
        description="Clear Files", button_style="warning",
        layout=L(width="110px", height="36px")
    )
    file_status = widgets.HTML(
        "<span style='color:#64748b;font-size:12px;'>No files uploaded.</span>"
    )

    # Chat area
    chat_widget = widgets.HTML(
        "<div class='ragbox'><div style='color:#475569;text-align:center;"
        "padding:60px 0 40px;'>Upload documents, click Index, then ask a question.</div></div>"
    )

    # Input row
    q_input = widgets.Text(
        placeholder="Ask a question about your documents…",
        layout=L(flex="1", height="38px")
    )
    send_btn = widgets.Button(
        description="Send",
        layout=L(width="80px", height="38px"),
        style={"button_color":"#4f46e5", "font_weight":"bold"},
    )
    new_chat_btn = widgets.Button(
        description="New Chat", button_style="warning",
        layout=L(width="90px", height="38px")
    )

    return dict(
        header=header, accordion=accordion,
        temp_sl=temp_sl, k_sl=k_sl, mem_sl=mem_sl, chunk_sl=chunk_sl,
        upload_btn=upload_btn, index_btn=index_btn,
        clear_files_btn=clear_files_btn, file_status=file_status,
        chat_widget=chat_widget,
        q_input=q_input, send_btn=send_btn, new_chat_btn=new_chat_btn,
    )


# ── Event handlers ────────────────────────────────────────────────────────────
def _on_index(_, w):
    global engine, processor, memory

    if not w["upload_btn"].value:
        w["file_status"].value = (
            "<span style='color:#f59e0b;font-size:12px;'>"
            "Select files using the Upload button first.</span>"
        )
        return

    w["index_btn"].disabled    = True
    w["index_btn"].description = "Indexing…"
    w["file_status"].value = (
        "<span style='color:#facc15;font-size:12px;'>Building index…</span>"
    )

    k        = w["k_sl"].value
    mem_win  = w["mem_sl"].value
    temp     = w["temp_sl"].value / 10
    chunk_sz = w["chunk_sl"].value

    try:
        # Write uploaded bytes to the session temp folder
        new_docs = []
        saved    = []
        for fname, fdata in w["upload_btn"].value.items():
            dest = os.path.join(SESSION_DIR, fname)
            with open(dest, "wb") as fh:
                fh.write(fdata["content"])
            saved.append(fname)
        # Re-create processor with current settings then load all session files
        processor = DocumentProcessor(
            chunk_size=chunk_sz, chunk_overlap=chunk_sz // 5, retriever_k=k
        )
        for f in Path(SESSION_DIR).iterdir():
            if f.is_file() and f.suffix.lower() in DocumentProcessor.LOADER_MAP:
                processor.load_file(str(f))

        processor.build_vector_store()
        memory = SessionMemory(window_k=mem_win)
        engine = RAGEngine(processor.get_retriever(), memory, temperature=temp)

        total    = len(processor.loaded_files)
        pills    = "".join(
            f"<span class='pill'>📄 {n}</span>" for n in processor.loaded_files
        )
        w["file_status"].value = (
            f"<span style='color:#4ade80;font-size:12px;font-weight:600;'>"
            f"Indexed {total} file(s):</span> {pills}"
        )
        chat_html_parts.clear()
        _push_bot(
            f"Knowledge base ready ({total} document(s) indexed).\n"
            f"Files: {', '.join(processor.loaded_files)}\n\n"
            "Ask me anything about your documents.",
            [],
            w["chat_widget"],
        )

    except Exception as exc:
        w["file_status"].value = (
            f"<span style='color:#f87171;font-size:12px;'>Error: {exc}</span>"
        )
        _push_err(str(exc), w["chat_widget"])
    finally:
        w["index_btn"].disabled    = False
        w["index_btn"].description = "Index Documents"


def _on_send(_, w):
    question = w["q_input"].value.strip()
    if not question:
        return
    if engine is None:
        _push_err(
            "No documents indexed. Upload files and click Index Documents.",
            w["chat_widget"]
        )
        return

    w["q_input"].value      = ""
    w["send_btn"].disabled  = True
    w["send_btn"].description = "…"
    _push_user(question, w["chat_widget"])

    try:
        result = engine.ask(question)
        _push_bot(result["answer"], result["sources"], w["chat_widget"])
    except Exception as exc:
        _push_err(str(exc), w["chat_widget"])
    finally:
        w["send_btn"].disabled    = False
        w["send_btn"].description = "Send"


def _on_new_chat(_, w):
    if engine:
        engine.memory.clear()
    chat_html_parts.clear()
    _render(w["chat_widget"])


def _on_clear_files(_, w):
    global processor, engine, memory
    for f in Path(SESSION_DIR).iterdir():
        f.unlink(missing_ok=True)
    processor = DocumentProcessor()
    memory    = SessionMemory()
    engine    = None
    chat_html_parts.clear()
    w["file_status"].value = (
        "<span style='color:#64748b;font-size:12px;'>All files cleared.</span>"
    )
    _render(w["chat_widget"])


# ── App layout & launch ───────────────────────────────────────────────────────
def launch_chatbot():
    """Render the full chatbot UI inline in the notebook."""
    w  = _make_widgets()
    L  = widgets.Layout

    # Wire events
    w["index_btn"].on_click(lambda b: _on_index(b, w))
    w["send_btn"].on_click(lambda b: _on_send(b, w))
    w["new_chat_btn"].on_click(lambda b: _on_new_chat(b, w))
    w["clear_files_btn"].on_click(lambda b: _on_clear_files(b, w))

    div  = lambda label: widgets.HTML(
        f"<div style='color:#818cf8;font-size:12px;font-weight:700;"
        f"letter-spacing:.05em;margin:10px 0 4px;text-transform:uppercase;'>{label}</div>"
    )
    sep  = widgets.HTML("<hr style='border-color:#1e1e3a;margin:6px 0;'>")

    app = widgets.VBox(
        [
            w["header"],
            w["accordion"],
            sep,
            div("Document Manager"),
            widgets.HBox(
                [w["upload_btn"], w["index_btn"], w["clear_files_btn"]],
                layout=L(gap="8px", align_items="center")
            ),
            w["file_status"],
            sep,
            div("Chat"),
            w["chat_widget"],
            widgets.HBox(
                [w["q_input"], w["send_btn"], w["new_chat_btn"]],
                layout=L(gap="8px", align_items="center", margin="6px 0 0")
            ),
        ],
        layout=L(
            width="820px", padding="18px",
            border="1px solid #1e1e3a", border_radius="14px",
        )
    )

    display(HTML(CSS))
    display(app)


print("Chat UI ready. Run launch_chatbot() in the next cell.")


Session folder: C:\Users\karel\AppData\Local\Temp\rag_session_0vru3n1_
Chat UI ready. Run launch_chatbot() in the next cell.


---
## Cell 5 — Launch

In [5]:
launch_chatbot()

---
## Cell 6 — Session Cleanup

Run this cell before closing the notebook to permanently delete all uploaded files from the temporary session folder.

Support for third party widgets will remain active for the duration of the session. To disable support:

In [6]:
import shutil, os

if os.path.exists(SESSION_DIR):
    shutil.rmtree(SESSION_DIR)
    print(f"Session folder deleted: {SESSION_DIR}")
else:
    print("Session folder already removed.")

Session folder deleted: C:\Users\karel\AppData\Local\Temp\rag_session_0vru3n1_


---
## Cell 7 — (Optional) CLI Testing

Tests the RAG pipeline directly — no UI required. Useful for validating a new document set or debugging prompt output.

In [7]:
# Uncomment and edit to run

DATA_DIR = "../data/files"       # path to a folder containing your documents

proc = DocumentProcessor(chunk_size=1000, chunk_overlap=200, retriever_k=4)
proc.load_directory(DATA_DIR)
proc.build_vector_store()

mem    = SessionMemory(window_k=5)
engine = RAGEngine(proc.get_retriever(), mem)

questions = [
    "What is a emebedding ?",
    "List out technical skill of Harshil Kareliya.",
    "What is RAG and why is it useful?",
]

for q in questions:
    print("\n" + "─" * 60)
    print(f"Q: {q}")
    r = engine.ask(q)
    print(f"A: {r['answer']}")
    print(f"Sources: {r['sources']}")

print("CLI test cell — uncomment the block above to use.")

  Loaded: 01_LLM_Theory_Notes.md
  Loaded: AI Sample questions.docx
  Loaded: embedding.pdf
  Loaded: Harshil_Kareliya_Resume2.docx
  Loaded: yolo.pdf


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  Indexing 203 chunks...
  Vector store ready.

────────────────────────────────────────────────────────────
Q: What is a emebedding ?
A: Based on the provided documents, an embedding (specifically, text embedding models) is defined as follows:

* **Definition:** It transforms natural language text, or textual and multimodal data, into mathematical vector representations.
* **Purpose and Applications:** These representations are used for knowledge base construction and play an indispensable role in text mining, question-answering systems, recommendation systems, and retrieval-augmented generation (RAG), significantly enhancing LLM-based agent systems.

*(Source: embedding.pdf, Page: 0)*
Sources: ['embedding.pdf (Page 15)', 'embedding.pdf (Page 4)', 'embedding.pdf (Page 17)', 'embedding.pdf (Page 0)']

────────────────────────────────────────────────────────────
Q: List out technical skill of Harshil Kareliya.
A: Based on the document `Harshil_Kareliya_Resume2.docx`, the technical skill

---

## Project Documentation

### 1. Problem Domain

Enterprise knowledge is locked inside documents — research papers, product manuals, meeting notes, policy files — that employees must search through manually. This chatbot addresses that problem by creating a **private, queryable knowledge base** from any uploaded document collection. Users get direct, cited answers rather than a list of documents to read.

The system is intentionally model-agnostic at the architectural level: swapping the LLM or embedding model requires changing only the provider class in `RAGEngine` and `DocumentProcessor`.

---

### 2. RAG Pipeline

#### 2.1 Document Ingestion

Supported formats are loaded through LangChain's document loader abstraction:

| Format | Loader |
|--------|--------|
| PDF    | `PyPDFLoader` — page-by-page text extraction |
| DOCX   | `Docx2txtLoader` — paragraph and table text |
| TXT / MD | `TextLoader` — raw UTF-8 text |

Each loaded document is tagged with `metadata["source_file"]` so the retrieval step can surface provenance alongside the answer.

#### 2.2 Chunking

`RecursiveCharacterTextSplitter` divides documents into overlapping chunks (default: 1000 characters, 200-character overlap). Overlapping windows ensure that sentences split across chunk boundaries are still retrieved coherently.

Separator hierarchy: `["\n\n", "\n", " ", ""]` — paragraph breaks are preferred over line breaks, which are preferred over word breaks.

#### 2.3 Embedding Generation

Chunks are embedded using **Google's `gemini-embedding-001`** model via `GoogleGenerativeAIEmbeddings`. This model produces 1536-dimensional dense vectors and ranks among the top performers on the MTEB retrieval benchmark.

The embedding model is instantiated once and cached inside `DocumentProcessor` to avoid repeated authentication overhead.

#### 2.4 Vector Store Indexing

Embedded chunks are stored in a **FAISS** (Facebook AI Similarity Search) flat index. FAISS performs exact nearest-neighbour search over the full corpus — appropriate for document sets up to several hundred thousand chunks without requiring an external database.

The `add_documents()` method supports incremental indexing: new files are embedded and merged into the existing FAISS index without a full rebuild.

#### 2.5 Query Processing & Retrieval

At query time:
1. The user question is embedded using the same `gemini-embedding-001` model.
2. FAISS returns the top-K most similar chunks (default K = 4) using cosine similarity.
3. Chunks are labelled with their source file and concatenated into a context string.

#### 2.6 Prompt Assembly & LLM Response

The final prompt is assembled from three components:

```
CONVERSATION HISTORY   ← last N human/AI turns (sliding window)
CONTEXT FROM DOCUMENTS ← top-K retrieved chunks with source labels
QUESTION               ← current user input
```

This assembled prompt is sent to **Gemini 1.5 Flash** via `ChatGoogleGenerativeAI`. The model is instructed to answer from the context first and fall back to general knowledge with an explicit prefix if the documents are insufficient.

The response and the original question are immediately saved to `SessionMemory` so subsequent questions can refer to prior answers.

---

### 3. Chatbot Interface

The UI is built entirely with **ipywidgets**, making it self-contained within a single `.ipynb` file — no Streamlit server, no separate process.

| Widget | Purpose |
|--------|---------|
| `FileUpload` | Upload PDF/DOCX/TXT/MD directly from the browser |
| Index button | Triggers `DocumentProcessor` pipeline and creates the FAISS index |
| Chat box | Scrollable HTML area with styled user/assistant bubbles |
| Text input | Submit with Enter key or the Send button |
| New Chat | Clears conversation memory without losing the document index |
| Clear Files | Wipes the temp folder and resets all state |
| Settings accordion | Live sliders for temperature, retrieval K, memory window, chunk size |

Uploaded files are written to `tempfile.mkdtemp()` — a system-managed temporary directory — which can be explicitly deleted by running Cell 6.

---

### 4. Design Choices

**Gemini for both LLM and embeddings** — Using the same provider for generation and embedding simplifies the dependency tree and the API key setup. The `gemini-embedding-001` model is free-tier eligible and outperforms many open-source alternatives on retrieval benchmarks.

**FAISS over hosted vector databases** — Chroma, Weaviate, and Qdrant require running an external service or connecting to a cloud endpoint. FAISS runs entirely in-process, making the notebook fully self-contained and offline-capable after the initial embedding API call.

**Sliding window memory over full-history injection** — Injecting the entire conversation history grows the prompt without bound. A configurable window (default: last 5 turns) keeps the token count predictable while still supporting follow-up questions.

**`InMemoryChatMessageHistory` over `langchain.memory`** — The `langchain.memory` module was deprecated and removed in LangChain ≥ 0.3. This project uses `langchain_core.chat_history.InMemoryChatMessageHistory`, which is the current stable API and has no additional dependencies.

---

### 5. How to Run

**Prerequisites**
- Python 3.10+
- `uv` (recommended) or `pip`
- A free Gemini API key from [Google AI Studio](https://aistudio.google.com)

**Steps**
```
1. Open RAG_Chatbot_Advanced.ipynb in Jupyter Lab / Notebook / VS Code
2. Run Cell 1  — installs all dependencies via uv (pip fallback)
3. Run Cell 2  — enter your Gemini API key when prompted
              (or create .env with GEMINI_API_KEY=... to skip this)
4. Run Cell 3  — loads the RAG pipeline classes
5. Run Cell 4  — loads the UI code
6. Run Cell 5  — launch_chatbot() renders the interactive UI
7. In the UI:
     a. Click Upload Files → select one or more documents
     b. Click Index Documents → wait for the FAISS index to build
     c. Type a question and press Enter or click Send
8. Run Cell 6  — deletes the temp session folder when you are done
```

**Optional: configure settings before indexing**
- Expand the *Advanced Settings* accordion to adjust temperature, retrieval K, memory window, and chunk size before clicking Index Documents.
- Changing settings and re-clicking Index will rebuild the index with the new parameters.

---

### 6. Project File Structure

```
project/
├── RAG_Chatbot_Advanced.ipynb   ← this notebook (single submission file)
├── .env                         ← optional — GEMINI_API_KEY=...
└── data/                        ← optional — for CLI test mode (Cell 7)
    └── your_documents.*
```

---

### 7. Dependencies

| Package | Role |
|---------|------|
| `langchain ≥ 0.3` | RAG orchestration framework |
| `langchain-google-genai` | Gemini LLM + embedding integration |
| `langchain-community` | FAISS vector store wrapper, document loaders |
| `langchain-text-splitters` | `RecursiveCharacterTextSplitter` |
| `faiss-cpu` | In-process vector similarity search |
| `pypdf` | PDF text extraction |
| `docx2txt` | DOCX text extraction |
| `google-generativeai` | Gemini API client |
| `python-dotenv` | `.env` file loading |
| `ipywidgets ≥ 8.0` | In-notebook interactive UI |
